# IMDB Sentiment Analysis — Exploratory Data Analysis
**Pinnacle Labs · Data Science Internship · Project 2**

This notebook covers:
1. Dataset overview & quality checks
2. Class distribution
3. Review length analysis
4. Most frequent words (raw vs cleaned)
5. Word clouds by sentiment
6. Sample reviews

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

sys.path.insert(0, '../src')
from preprocess import clean_text, STOP_WORDS

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

DATA_PATH = '../data/IMDB Dataset.csv'
print('Imports done.')

## 1. Load & Overview

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape      : {df.shape}')
print(f'Columns    : {list(df.columns)}')
print(f'Null values:\n{df.isnull().sum()}')
print(f'Duplicates : {df.duplicated().sum()}')
df.head(3)

In [ ]:
df.info()

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['sentiment'].value_counts()

# Bar chart
axes[0].bar(counts.index, counts.values, color=['#28a745', '#dc3545'], edgecolor='white', linewidth=0.5)
axes[0].set_title('Review Count by Sentiment', fontweight='bold')
axes[0].set_ylabel('Count')
for i, (idx, v) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(
    counts.values,
    labels     = counts.index,
    autopct    = '%1.1f%%',
    colors     = ['#28a745', '#dc3545'],
    startangle = 90,
    wedgeprops = {'edgecolor': 'white', 'linewidth': 2},
)
axes[1].set_title('Sentiment Proportion', fontweight='bold')

plt.suptitle('Dataset is Perfectly Balanced (25K each)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../models/eda_class_distribution.png', bbox_inches='tight')
plt.show()
print(counts)

## 3. Review Length Analysis

In [ ]:
df['char_count']  = df['review'].str.len()
df['word_count']  = df['review'].str.split().str.len()

print('--- Character Count ---')
print(df.groupby('sentiment')['char_count'].describe().round(1))
print('\n--- Word Count ---')
print(df.groupby('sentiment')['word_count'].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for sentiment, color in [('positive', '#28a745'), ('negative', '#dc3545')]:
    subset = df[df['sentiment'] == sentiment]
    axes[0].hist(subset['word_count'].clip(upper=800), bins=60, alpha=0.6,
                 label=sentiment.capitalize(), color=color, edgecolor='none')
    axes[1].hist(subset['char_count'].clip(upper=6000), bins=60, alpha=0.6,
                 label=sentiment.capitalize(), color=color, edgecolor='none')

axes[0].set_title('Word Count Distribution', fontweight='bold')
axes[0].set_xlabel('Number of words')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].axvline(df['word_count'].median(), color='black', linestyle='--', alpha=0.5, label=f'Median: {df["word_count"].median():.0f}')

axes[1].set_title('Character Count Distribution', fontweight='bold')
axes[1].set_xlabel('Number of characters')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.suptitle('Review Length by Sentiment', fontsize=13)
plt.tight_layout()
plt.savefig('../models/eda_length_distribution.png', bbox_inches='tight')
plt.show()

## 4. Preprocessing Sample

In [ ]:
sample_reviews = df.sample(3, random_state=42)['review'].values
for i, r in enumerate(sample_reviews):
    print(f'--- Review {i+1} ---')
    print(f'RAW   : {r[:200]}...')
    print(f'CLEAN : {clean_text(r)[:200]}...')
    print()

In [ ]:
# Run on a sample of 2000 to compare before/after lengths
sample_df = df.sample(2000, random_state=42).copy()
sample_df['clean_review'] = sample_df['review'].apply(clean_text)
sample_df['clean_word_count'] = sample_df['clean_review'].str.split().str.len()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(sample_df['word_count'].clip(upper=600),       bins=50, alpha=0.7, label='Before cleaning', color='#6C63FF')
ax.hist(sample_df['clean_word_count'].clip(upper=600), bins=50, alpha=0.7, label='After cleaning',  color='#FF6584')
ax.set_title('Word Count Before vs After Preprocessing (sample=2000)', fontweight='bold')
ax.set_xlabel('Word count')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.savefig('../models/eda_cleaning_comparison.png', bbox_inches='tight')
plt.show()

reduction = (1 - sample_df['clean_word_count'].mean() / sample_df['word_count'].mean()) * 100
print(f'Average word reduction after cleaning: {reduction:.1f}%')

## 5. Most Frequent Words by Sentiment

In [ ]:
# Use sample_df (already has clean_review)
pos_words = ' '.join(sample_df[sample_df['sentiment'] == 'positive']['clean_review']).split()
neg_words = ' '.join(sample_df[sample_df['sentiment'] == 'negative']['clean_review']).split()

top_pos = Counter(pos_words).most_common(20)
top_neg = Counter(neg_words).most_common(20)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Positive
words_p, counts_p = zip(*top_pos)
axes[0].barh(list(reversed(words_p)), list(reversed(counts_p)), color='#28a745', edgecolor='none')
axes[0].set_title('Top 20 Words — Positive Reviews', fontweight='bold')
axes[0].set_xlabel('Frequency')

# Negative
words_n, counts_n = zip(*top_neg)
axes[1].barh(list(reversed(words_n)), list(reversed(counts_n)), color='#dc3545', edgecolor='none')
axes[1].set_title('Top 20 Words — Negative Reviews', fontweight='bold')
axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('../models/eda_top_words.png', bbox_inches='tight')
plt.show()

## 6. Word Clouds

In [ ]:
pos_text = ' '.join(sample_df[sample_df['sentiment'] == 'positive']['clean_review'])
neg_text = ' '.join(sample_df[sample_df['sentiment'] == 'negative']['clean_review'])

def make_wordcloud(text, colormap, title, ax):
    wc = WordCloud(
        width=600, height=400, background_color='white',
        colormap=colormap, max_words=150,
        collocations=False,
    ).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
make_wordcloud(pos_text, 'Greens',  'Positive Reviews Word Cloud', axes[0])
make_wordcloud(neg_text, 'Reds',    'Negative Reviews Word Cloud', axes[1])

plt.suptitle('Word Clouds by Sentiment', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/eda_wordclouds.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Sample Reviews Showcase

In [ ]:
print('=== POSITIVE REVIEWS (sample) ===')
for txt in df[df['sentiment']=='positive']['review'].sample(3, random_state=1).values:
    print(f'• {txt[:300]}\n')

print('=== NEGATIVE REVIEWS (sample) ===')
for txt in df[df['sentiment']=='negative']['review'].sample(3, random_state=1).values:
    print(f'• {txt[:300]}\n')

## Summary

| Insight | Finding |
|---|---|
| Dataset size | 50,000 reviews |
| Class balance | Perfectly balanced (50% / 50%) |
| Avg word count | ~230 words per review |
| After cleaning | ~40-50% word reduction |
| Key positive words | great, good, best, love, excellent |
| Key negative words | bad, worst, waste, boring, awful |

**Next:** Run `python src/train.py` to train models.